In [1]:
import pandas as pd
import numpy as np
import re


import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df=pd.read_csv("/content/twitter_training.csv",header=None)
df.columns=["id","topic","sentiment","text"]


In [3]:
df=df[df['sentiment']!= 'Irrelevant']


In [4]:
df['sentiment']=df['sentiment'].astype(str)

In [5]:
df['sentiment']=df['sentiment'].str.strip()

In [6]:
df["sentiment"]=df['sentiment'].map({
    'Positive':1,
    'Negative':0,
    'Neutral':2
})

In [7]:
df.dropna(inplace=True)

In [8]:
print(df['sentiment'].value_counts())

sentiment
0    22358
1    20655
2    18108
Name: count, dtype: int64


In [9]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

def preprocess_text(text):
    text=str(text)

    text=text.lower()
    text=re.sub(r'http\S+|www\S+','',text)
    text=re.sub(r'@\w+','',text)
    text=re.sub(r'[^a-zA-Z\s]','',text)
    tokens=text.split()
    tokens=[lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return" ".join(tokens)
df['clean_text']=df['text'].apply(preprocess_text)

In [10]:
df=df.sample(frac=1,random_state=42)
x=df['clean_text']
y=df['sentiment']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [11]:
tfidf=TfidfVectorizer(max_features=5000)
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [12]:
lr=LogisticRegression(max_iter=200)
lr.fit(x_train_tfidf,y_train)
y_pred_lr=lr.predict(x_test_tfidf)

In [16]:
nb=MultinomialNB()
nb.fit(x_train_tfidf,y_train)
y_pred_nb=nb.predict(x_test_tfidf)

In [17]:
dt=DecisionTreeClassifier()
dt.fit(x_train_tfidf,y_train)
y_pred_dt=dt.predict(x_test_tfidf)

In [18]:
def evaluate_model(y_test,y_pred,name):
    print(f"\n{name}")
    print("Accuracy:",accuracy_score(y_test,y_pred))
    print("Precision:",precision_score(y_test,y_pred,average='weighted'))
    print("Recall:",recall_score(y_test,y_pred,average='weighted'))
    print("F1 Score:",f1_score(y_test,y_pred,average='weighted'))

In [19]:
evaluate_model(y_test,y_pred_lr,"Logistic Regression")
evaluate_model(y_test,y_pred_nb,"Naive Bayes")
evaluate_model(y_test,y_pred_dt,"Decision Tree")


Logistic Regression
Accuracy: 0.7569734151329244
Precision: 0.7563325280875858
Recall: 0.7569734151329244
F1 Score: 0.7565790185729634

Naive Bayes
Accuracy: 0.7198364008179959
Precision: 0.7238371501224399
Recall: 0.7198364008179959
F1 Score: 0.7146887136291874

Decision Tree
Accuracy: 0.8086707566462168
Precision: 0.8099454055765865
Recall: 0.8086707566462168
F1 Score: 0.8085537270548878


In [20]:
results=pd.DataFrame({
    'Model':['Logistic Regression','Naive Bayes','Decision tree'],
    'Accuracy':[
        accuracy_score(y_test,y_pred_lr),
        accuracy_score(y_test,y_pred_nb),
        accuracy_score(y_test,y_pred_dt)
    ],
    'F1 Score':[
        f1_score(y_test,y_pred_lr,average='weighted'),
        f1_score(y_test,y_pred_nb,average='weighted'),
        f1_score(y_test,y_pred_dt,average='weighted')

    ]
})
print("\nComparision Table:")
print(results)


Comparision Table:
                 Model  Accuracy  F1 Score
0  Logistic Regression  0.756973  0.756579
1          Naive Bayes  0.719836  0.714689
2        Decision tree  0.808671  0.808554


In [21]:
print("\nComparision Report(Logistic Regression):")
print(classification_report(y_test,y_pred_lr))


Comparision Report(Logistic Regression):
              precision    recall  f1-score   support

           0       0.79      0.80      0.80      4472
           1       0.76      0.77      0.77      4131
           2       0.71      0.69      0.70      3622

    accuracy                           0.76     12225
   macro avg       0.75      0.75      0.75     12225
weighted avg       0.76      0.76      0.76     12225

